# Refgenie tutorial

The `Refgenie` class is a key component of the Refgenie package, which is used for managing and organizing reference genome files. It provides a set of methods and functionalities to interact with reference genome assets, and other related resources.

## Purpose of this file

This file serves as a tutorial for using the `Refgenie` Python API. It demonstrates how to set up a temporary directory for storing reference genome assets, configure the `Refgenie` instance, and perform various operations such as listing available assets, retrieving asset information, and managing data channels. 

In order to learn more about any of the contepts indicated in the code, please refer to a specific section of the documentation.

## Installation

Before the package gets released, clone the repository install, for example using `uv`:
  
```bash
git clone <repo_url>
cd refgenie1
uv pip install .
```

## Configuration

First, let's create a temporary directory that will be used to store the refgenie assets.


In [1]:
from pathlib import Path
from rich import print
import os

Let's set a temporary directory to store the refgenie assets.

In [2]:
from tempfile import TemporaryDirectory

REFGENIE_CODE_PATH = Path.cwd().parent / "refgenie"

# set the environment variable
archive_tmp_dir = TemporaryDirectory(prefix="refgenie_archive_demo_").name
os.environ["REFGENIE_GENOME_STAGE_FOLDER"] = archive_tmp_dir
tmp_dir = TemporaryDirectory(prefix="refgenie_demo_").name
os.environ["REFGENIE_GENOME_FOLDER"] = tmp_dir
# set the REFGENIE_DB_CONFIG_PATH to a sqlite config file in the refgenie package
os.environ["REFGENIE_DB_CONFIG_PATH"] = (REFGENIE_CODE_PATH / "config" / "sqlite_config.yaml").as_posix()


Let's inspect the refgenie configuration object.

In [3]:
from refgenie.config import config

print(config)

RefgenieConfig(
    log_level=<LogLevel.INFO: 'INFO'>,
    genome_folder=PosixPath('/tmp/refgenie_demo_2hzk45es'),
    genome_stage_folder=PosixPath('/tmp/refgenie_archive_demo_3ps6xfvv'),
    database_config_path=PosixPath('/home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/config/sqlite
_config.yaml'),
    bridge_mode='read',
    bridge_origins='https://refgenie.org,https://ui.refgenie.org',
    bridge_origin_regex='',
    bridge_expose_paths=False
)

### Database backend

As you can see, refgenie configuration points to a database configuration file, as by default refgenie is backed by a SQLite database.

Let's inspect the refgenie database configuration file.

In [4]:
%cat {config.database_config_path}

type: sqlite
path: ~/refgenie_db/refgenie


Make sure the directory where the SQLite database file is stored exists, and create it if it doesn't.

In [5]:
!refgenie purge --force
!rm -rf ~/refgenie_db
!mkdir -p ~/refgenie_db

Traceback (most recent call last):
  File "/home/nsheff/.local/bin/refgenie", line 5, in <module>
    from refgenie.cli.cli_pydantic import main_cli
ModuleNotFoundError: No module named 'refgenie.cli.cli_pydantic'


In practice, you don't even need to create the configuration file manually, as refgenie ships with a default configuration file that is used if no configuration file is provided. Just as we've seen above.

For production deployments you may want to use a different database backend, such as MySQL or PostgreSQL. In this case, you can provide the database configuration file path by setting `REFGENIE_DB_CONFIG_PATH` environment variable, or even set/override the database engine using `database_engine` in the `Refgenie` constructor. The object must be a `sqlalchemy.engine.Engine` object.



### Refgenieserver client

Similarly, refgenie ships with a Refgenieserver client, which is used by default to retrieve remote genome assets and does not need to be replaced in majority of use cases. However, you can provide a custom URL-client mapping to `Refgenie` constructor, by setting the `server_client_mapping` argument. Please note that, the clients need to follow a specific interface, defined in `refgenie.server.ServerClient` protocol. More details below.

In [6]:
from refgenie.managers.sources.client import ServerClient
from rich import inspect

inspect(
    ServerClient,
    methods=True,
    docs=True,
    help=True,
    title="ServerClient Protocol structure",
)

╭──────────────────────────────────────── ServerClient Protocol structure ────────────────────────────────────────╮
│ class ServerClient(*args, **kwargs):                                                                            │
│                                                                                                                 │
│ Protocol for the server client.                                                                                 │
│                                                                                                                 │
│             server_url = <property object at 0x7ad9be7b03b0>                                                    │
│          download_file = def download_file(self, asset_digest: str, file_path: str, output_path: pathlib.Path)  │
│                          -> pathlib.Path: Download a single file from a file-mode asset.                        │
│ download_with_progress = def download_with_progress(self, operation_id: str, output_path: pathlib.Path, params: │
│                          dict | None = None, url_format_params: dict[str, str | None] = None, name: str | None  │
│                          = None) -> pathlib.Path:                                                               │
│                          Download the asset served by the given operation_id to output_path,                    │
│                          showing progress along the way.                                                        │
│                                                                                                                 │
│                          Implementations should report byte progress to `refgenie.progress`                     │
│                          whenever `refgenie.progress.active_sink()` is not None, and should not                 │
│                          build a `rich` live display in that case -- something other than a                     │
│                          terminal is consuming the progress.                                                    │
│                    get = def get(self, operation_id: str, params: dict | None = None, url_format_params:        │
│                          dict[str, str | None] = None) -> dict: Send a GET request to the specified operation   │
│                          ID.                                                                                    │
│        get_all_aliases = def get_all_aliases(self, params: dict | None = None) -> list[dict]: Get all aliases   │
│                          using pagination.                                                                      │
│   get_all_asset_groups = def get_all_asset_groups(self, params: dict | None = None) -> list[dict]: Get all      │
│                          asset groups using pagination.                                                         │
│         get_all_assets = def get_all_assets(self, params: dict | None = None) -> list[dict]: Get all assets     │
│                          using pagination.                                                                      │
│        get_all_genomes = def get_all_genomes(self, params: dict | None = None) -> list[dict]: Get all genomes   │
│                          using pagination.                                                                      │
│  get_all_staged_assets = def get_all_staged_assets(self, params: dict | None = None) -> list[dict]: Get all     │
│                          staged assets using pagination.                                                        │
│    get_asset_file_list = def get_asset_file_list(self, asset_digest: str) -> list[str]: Get the list of files   │
│                          available for a file-mode asset.                                                       │
│       get_asset_groups = def get_asset_groups(self, params: dict | None = None) -> list[dict]: Get one page of  │
│                          asset groups, unwrapping the 

First, let's import the `Refgenie` class from the `refgenie` package.


In [7]:
from refgenie import Refgenie

refgenie = Refgenie(suppress_migrations=True)

Let's ensure we start with a clean slate by removing any existing refgenie metadata and initializing a new refgenie instance.


In [8]:
refgenie.init()  # initialize new refgenie instance

INFO     Genome folder ready: /tmp/refgenie_demo_2hzk45es                                          ]8;id=412786;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py\lifecycle.py]8;;\:]8;id=350506;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py#182\182]8;;\

INFO     Genome stage folder ready: /tmp/refgenie_archive_demo_3ps6xfvv                            ]8;id=967050;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py\lifecycle.py]8;;\:]8;id=475640;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py#186\186]8;;\

INFO     Initialized refgenie backend: 'sqlite:////home/nsheff/refgenie_db/refgenie'               ]8;id=656192;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py\lifecycle.py]8;;\:]8;id=373822;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/core/lifecycle.py#106\106]8;;\

Before we can pull or build a `fasta` asset, refgenie needs to know the `fasta` asset class and recipe. Let's register them from the recipes repository (we'll add a whole data channel of recipes later, but for now we just need `fasta`):

In [9]:
refgenie.asset_class.add(
    "https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/fasta_asset_class.yaml"
)
refgenie.recipe.add(
    "https://github.com/refgenie/recipes/raw/refgenie1/recipes/fasta_asset_recipe.yaml"
)

INFO     Reading YAML from URL:                                                                            ]8;id=876935;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=937388;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/fasta_asset_class.yaml                    

INFO     HTTP Request: GET                                                                          ]8;id=719802;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=766805;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/fasta_asset_class.yaml                    
         "HTTP/1.1 302 Found"                                                                                      

INFO     HTTP Request: GET                                                                          ]8;id=986204;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=192257;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://raw.githubusercontent.com/refgenie/recipes/refgenie1/asset_classes/fasta_asset_cla                
         ss.yaml "HTTP/1.1 200 OK"                                                                                 

INFO     Registered 'fasta' asset class                                                           ]8;id=514958;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py\asset_class.py]8;;\:]8;id=364717;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py#90\90]8;;\

INFO     Reading YAML from URL:                                                                            ]8;id=120478;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=350303;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/recipes/fasta_asset_recipe.yaml                         

INFO     HTTP Request: GET                                                                          ]8;id=669243;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=234701;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/recipes/fasta_asset_recipe.yaml                         
         "HTTP/1.1 302 Found"                                                                                      

INFO     HTTP Request: GET                                                                          ]8;id=584837;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=858600;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://raw.githubusercontent.com/refgenie/recipes/refgenie1/recipes/fasta_asset_recipe.ya                
         ml "HTTP/1.1 200 OK"                                                                                      

INFO     Registered 'fasta' recipe                                                                    ]8;id=550971;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py\recipe.py]8;;\:]8;id=437704;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py#140\140]8;;\

Recipe(id=1, name='fasta', version='0.1.0', description='DNA sequences in the FASTA format, exported from RefgetStore. Includes FASTA index (.fai) and chromosome sizes file. Requires genome to be initialized in RefgetStore first (via refgenie genome init).', output_asset_class_id=1, command_templates=['refgenie-build-fasta {{values.refget_store_path}} {{values.genome_digest}} {{values.output_folder}}'], input_params=None, input_files=None, input_assets=None, docker_image=None, custom_seek_keys={}, default_asset='default', inherent=None, updated_at=datetime.datetime(2026, 8, 21, 1, 55, 43, 254318, tzinfo=datetime.timezone.utc), created_at=datetime.datetime(2026, 8, 21, 1, 55, 43, 254321, tzinfo=datetime.timezone.utc))

Let's subscribe to the default refgenie server. This method will reach out to the server at the provided URL and query the OpenAPI specification to determine whether ther server is refgenie-compatible. If it is, the server will be added to the list of subscribed servers.

Note: there's currently no public compatible refgenieserver instance deployed, so the following code snippets use a local refgenieserver instance serving the latest API.


In [10]:
refgenie.configuration.subscribe("http://localhost:8000")

INFO     Subscribed to servers: http://localhost:8000                                           ]8;id=766030;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/configuration.py\configuration.py]8;;\:]8;id=557829;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/configuration.py#55\55]8;;\

And that's it! We have now configured a refgenie instance and subscribed to a refgenie-compatible server. We can now start using the refgenie instance to manage reference genome assets.

### Pull an asset

Let's initialize a new genome by pulling an asset of fasta class. This will create a new directory in the `data` subdirectory of the `genome_folder` and mirror it in the `alias` directory with symbolic links, rather than copies of the files.


In [11]:
refgenie.pull(alias_name="rCRSd", asset_group_name="fasta")

INFO     HTTP Request: GET http://localhost:8000/openapi.json "HTTP/1.1 200 OK"                     ]8;id=352808;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=213390;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\

INFO     Connected to server: http://localhost:8000 title='Refgenieserver REST API' version='1.0.0a1' ]8;id=394534;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/sources/client.py\client.py]8;;\:]8;id=559133;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/sources/client.py#221\221]8;;\
         description='a web interface and RESTful API for reference genome assets'                                 

WARNING  No local digest for genome alias: rCRSd. Setting genome identity with server:                ]8;id=611532;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/puller.py\puller.py]8;;\:]8;id=145606;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/puller.py#452\452]8;;\
         http://localhost:8000                                                                                     

INFO     HTTP Request: GET http://localhost:8000/rgstore.json "HTTP/1.1 200 OK"                     ]8;id=658149;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=285831;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\

INFO     http://localhost:8000 is not a RefgetStore root; looking for a server service-info.         ]8;id=24266;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/sources/genomes.py\genomes.py]8;;\:]8;id=648393;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/sources/genomes.py#355\355]8;;\

...

INFO     Initializing genome from FASTA file:                                                         ]8;id=749550;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py\genome.py]8;;\:]8;id=46341;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py#417\417]8;;\
         /tmp/refgenie_demo_2hzk45es/data/jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP/fasta/03aa63e8accae38cf4d6              
         bcbb7865ef85f95a54020c4f4d48921e67ce382f188c/jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP.fa                          

INFO     Added: 'rCRSd/fasta:default'                                                                ]8;id=717211;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/content.py\content.py]8;;\:]8;id=707148;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/content.py#381\381]8;;\

INFO     Set default asset: 'jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP/fasta:default'                            ]8;id=302938;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/seek.py\seek.py]8;;\:]8;id=931030;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/seek.py#612\612]8;;\

INFO     Created alias directories: /tmp/refgenie_demo_2hzk45es/alias/rCRSd/fasta/default           ]8;id=368520;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=482744;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\

Asset(name='default', description=None, size=34113, colocate=None, updated_at=datetime.datetime(2026, 8, 21, 1, 55, 43, 893614), path='data/jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP/fasta/03aa63e8accae38cf4d6bcbb7865ef85f95a54020c4f4d48921e67ce382f188c', digest='03aa63e8accae38cf4d6bcbb7865ef85f95a54020c4f4d48921e67ce382f188c', recipe_id=None, serving_modes_override=None, asset_group_id=1, created_at=datetime.datetime(2026, 8, 21, 1, 55, 43, 893622))

As you can see above, the genome has been initialized and `fasta` asset was pulled. Let's inspect the initialized genome.


In [12]:
print(refgenie.genome.table())

                                       Genomes                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Digest                           ┃ Aliases ┃ Source ┃ Species ┃ Description       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP │ rCRSd   │ local  │         │ rCRSd test genome │
└──────────────────────────────────┴─────────┴────────┴─────────┴───────────────────┘

Now, that a `fasta` asset has been built for the `dm6` genome, let's add some custom asset classes and recipes to build an asset based on that.

### Add `bowtie2_index` asset class and recipe

By supplying a URL (`str` object) rather than a local path (`pathlib.Path` object), refgenie will grab the remote file and register it as if it was a local file.


In [13]:
refgenie.asset_class.add(
    "https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/bowtie2_index_asset_class.yaml"
)
refgenie.recipe.add(
    "https://github.com/refgenie/recipes/raw/refgenie1/recipes/bowtie2_index_asset_recipe.yaml"
)

INFO     Reading YAML from URL:                                                                            ]8;id=55015;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=135783;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/bowtie2_index_asset_class.yaml            

INFO     HTTP Request: GET                                                                          ]8;id=356260;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=732153;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/asset_classes/bowtie2_index_asset_class.                
         yaml "HTTP/1.1 302 Found"                                                                                 

INFO     HTTP Request: GET                                                                          ]8;id=732772;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=520565;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://raw.githubusercontent.com/refgenie/recipes/refgenie1/asset_classes/bowtie2_index_a                
         sset_class.yaml "HTTP/1.1 200 OK"                                                                         

INFO     Registered 'bowtie2_index' asset class                                                   ]8;id=673012;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py\asset_class.py]8;;\:]8;id=932813;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py#90\90]8;;\

INFO     Reading YAML from URL:                                                                            ]8;id=66012;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=549617;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/recipes/bowtie2_index_asset_recipe.yaml                 

INFO     HTTP Request: GET                                                                          ]8;id=228749;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=893175;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://github.com/refgenie/recipes/raw/refgenie1/recipes/bowtie2_index_asset_recipe.yaml                 
         "HTTP/1.1 302 Found"                                                                                      

INFO     HTTP Request: GET                                                                          ]8;id=392934;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=514465;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://raw.githubusercontent.com/refgenie/recipes/refgenie1/recipes/bowtie2_index_asset_r                
         ecipe.yaml "HTTP/1.1 200 OK"                                                                              

INFO     Registered 'bowtie2_index' recipe                                                            ]8;id=203095;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py\recipe.py]8;;\:]8;id=619891;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py#140\140]8;;\

Recipe(id=2, name='bowtie2_index', version='0.0.1', description='Genome index for bowtie2, produced with bowtie2-build', output_asset_class_id=2, command_templates=['bowtie2-build --threads {{values.params["threads"]}} {{values.genome_folder}}/{{values.assets["fasta"].seek_keys_dict["fasta"]}} {{values.output_folder}}/{{values.genome_digest}}'], input_params={'threads': {'description': 'Number of threads to use', 'default': 1}}, input_files=None, input_assets={'fasta': {'asset_class': 'fasta', 'description': 'fasta asset for genome', 'default': 'fasta'}}, docker_image='docker.io/databio/refgenie', custom_seek_keys={'version': "bowtie2-build --version | awk 'NR==1{print $3}'"}, default_asset='{{values.custom_seek_keys.version}}', inherent=None, updated_at=datetime.datetime(2026, 8, 21, 1, 55, 44, 426089, tzinfo=datetime.timezone.utc), created_at=datetime.datetime(2026, 8, 21, 1, 55, 44, 426090, tzinfo=datetime.timezone.utc))

Let's verify that it worked by listing the available asset classes and recipes:


In [14]:
from rich import print

print(refgenie.recipe.table())
print(refgenie.asset_class.table())

                                                      Recipes                                                      
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃               ┃         ┃ Output asset     ┃ Input asset     ┃             ┃                  ┃                 ┃
┃ Name          ┃ Version ┃ class            ┃ classes         ┃ Input files ┃ Input params     ┃ Docker image    ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ fasta         │ 0.1.0   │ fasta            │ None            │ None        │ None             │                 │
├───────────────┼─────────┼──────────────────┼─────────────────┼─────────────┼──────────────────┼─────────────────┤
│ bowtie2_index │ 0.0.1   │ bowtie2_index    │ • fasta (fasta  │ None        │ • threads        │ docker.io/data… │
│               │         │                  │ asset for       │             │ (Number of       │                 │
│               │         │                  │ genome)         │             │ threads to use)  │                 │
│               │         │                  │ default=fasta   │             │ default=1        │                 │
└───────────────┴─────────┴──────────────────┴─────────────────┴─────────────┴──────────────────┴─────────────────┘

                                                   Asset Classes                                                   
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name          ┃ Version ┃ Seek keys               ┃ Description                                                 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ fasta         │ 0.1.0   │ fasta, fai, chrom_sizes │ Sequences in the FASTA format, indexed FASTA and chromosome │
│               │         │                         │ sizes file                                                  │
├───────────────┼─────────┼─────────────────────────┼─────────────────────────────────────────────────────────────┤
│ bowtie2_index │ 0.0.1   │ bowtie2_index           │ Genome index for bowtie2, produced with bowtie2-build       │
└───────────────┴─────────┴─────────────────────────┴─────────────────────────────────────────────────────────────┘

## Build a `fasta` asset


In [15]:
from refgenie import BuildParams
from pathlib import Path

# Step 1: Initialize genome from FASTA
refgenie.genome.initialize_genome(
    fasta_file_path=REFGENIE_CODE_PATH.parent / "tests/data/t7.fa",
    alias_names=["t7"],
    description="Genome of T7 phage",
)

# Step 2: Build fasta asset (exports from RefgetStore, no input file needed)
refgenie.build_asset(
    recipe_name="fasta",
    genome_name="t7",
    asset_group_name="fasta",
)

INFO     Initializing genome from FASTA file:                                                         ]8;id=171856;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py\genome.py]8;;\:]8;id=934094;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py#417\417]8;;\
         /home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/tests/data/t7.fa                                 

INFO     Added genome: kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB                                                ]8;id=617785;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py\genome.py]8;;\:]8;id=998458;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/genome.py#90\90]8;;\

INFO     Added alias: t7                                                                               ]8;id=780321;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py\alias.py]8;;\:]8;id=271438;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py#305\305]8;;\

INFO     Building 't7/fasta' using recipe 'fasta (v0.1.0)'                                           ]8;id=802800;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py\builder.py]8;;\:]8;id=180848;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py#513\513]8;;\

INFO     Could not locate config file.                                                                ]8;id=868055;file:///home/nsheff/Dropbox/workspaces/pepkit/repos/yacman/yacman/yacman.py\yacman.py]8;;\:]8;id=455348;file:///home/nsheff/Dropbox/workspaces/pepkit/repos/yacman/yacman/yacman.py#674\674]8;;\

...

INFO     Added: 't7/fasta:default'                                                                   ]8;id=782410;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/content.py\content.py]8;;\:]8;id=115875;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/content.py#381\381]8;;\

INFO     Set default asset: 'kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/fasta:default'                            ]8;id=638527;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/seek.py\seek.py]8;;\:]8;id=301347;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/seek.py#612\612]8;;\

INFO     Added asset: 'kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/fasta:default'                               ]8;id=970752;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py\builder.py]8;;\:]8;id=233004;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py#705\705]8;;\

INFO     Created alias directories: /tmp/refgenie_demo_2hzk45es/alias/t7/fasta/default              ]8;id=14000;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=694343;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\

Asset(name='default', description='DNA sequences in the FASTA format, exported from RefgetStore. Includes FASTA index (.fai) and chromosome sizes file. Requires genome to be initialized in RefgetStore first (via refgenie genome init).', size=40486, colocate=None, updated_at=datetime.datetime(2026, 8, 21, 1, 55, 44, 814061), path='data/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/fasta/949c705a1c1668f0ea3d7da883d1a65e902b945b67de576407349ff72a8b1bf1', digest='949c705a1c1668f0ea3d7da883d1a65e902b945b67de576407349ff72a8b1bf1', recipe_id=1, serving_modes_override=None, asset_group_id=2, created_at=datetime.datetime(2026, 8, 21, 1, 55, 44, 814074))

### Build a `bowtie2_index` asset

The `bowtie2_index` asset class and recipe have been added successfully. Let's build the `bowtie2_index` asset for the `dm6` genome.


In [16]:
from refgenie.models import BuildParams

refgenie.build_asset(
    recipe_name="bowtie2_index",
    genome_name="t7",
    asset_group_name="bowtie2_index",
    params=BuildParams(params={"threads": 8}),
    stage=True,  # stage the asset right after building
)

INFO     Building 't7/bowtie2_index' using recipe 'bowtie2_index (v0.0.1)'                           ]8;id=468937;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py\builder.py]8;;\:]8;id=447964;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/builder.py#513\513]8;;\

[2026-08-21T01:55:44Z WARN  bulker::shimlink] 'bowtie2-build': `docker_command` is deprecated; use the `entrypoint` manifest field instead


INFO     Could not locate config file.                                                                ]8;id=518856;file:///home/nsheff/Dropbox/workspaces/pepkit/repos/yacman/yacman/yacman.py\yacman.py]8;;\:]8;id=599693;file:///home/nsheff/Dropbox/workspaces/pepkit/repos/yacman/yacman/yacman.py#674\674]8;;\

INFO     No output schema supplied. Running in schema-optional mode (validate_results=False).      ]8;id=459214;file:///home/nsheff/.local/lib/python3.12/site-packages/pipestat/pipestat.py\pipestat.py]8;;\:]8;id=33995;file:///home/nsheff/.local/lib/python3.12/site-packages/pipestat/pipestat.py#1008\1008]8;;\
         Results will not be validated against a schema.                                                           

INFO     Initializing results file                                                               ]8;id=423030;file:///home/nsheff/.local/lib/python3.12/site-packages/pipestat/backends/file_backend/filebackend.py\filebackend.py]8;;\:]8;id=556113;file:///home/nsheff/.local/lib/python3.12/site-packages/pipestat/backends/file_backend/filebackend.py#785\785]8;;\
         '/tmp/refgenie_demo_2hzk45es/builds/t7/bowtie2_index/2.3.5/stats.yaml'                                    

...

INFO     Updated parents of 'kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index:2.3.5'                 ]8;id=953039;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/relations.py\relations.py]8;;\:]8;id=884069;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/relations.py#166\166]8;;\

INFO     Staging asset: kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index:2.3.5                            ]8;id=668221;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py\stage.py]8;;\:]8;id=397506;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py#67\67]8;;\

INFO     Reconciled file-serving stage dir:                                                            ]8;id=984501;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py\stage.py]8;;\:]8;id=947814;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py#335\335]8;;\
         /tmp/refgenie_archive_demo_3ps6xfvv/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index/2.3.5 (6               
         link(s), 6 written or removed; excluded 0 colocation symlink(s))                                          

INFO     Created file-serving StagedAsset for kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index:2.3.5     ]8;id=170772;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py\stage.py]8;;\:]8;id=267267;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/stage.py#217\217]8;;\

Asset(name='2.3.5', description='Genome index for bowtie2, produced with bowtie2-build', size=8445686, colocate=None, updated_at=datetime.datetime(2026, 8, 21, 1, 55, 46, 9717), path='data/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index/2b4c1900ec5a2b9fce39e45a71e6bed38ca234d394b8af28c2a3b0e09548d51e', digest='2b4c1900ec5a2b9fce39e45a71e6bed38ca234d394b8af28c2a3b0e09548d51e', recipe_id=2, serving_modes_override=None, asset_group_id=3, created_at=datetime.datetime(2026, 8, 21, 1, 55, 46, 9733))

Let's list the assets for the genome `t7` to verify that the `bowtie2_index` asset has been built successfully.


In [17]:
refgenie.asset.table(genome_names=["t7"])[0]

                     Refgenie assets. Source: local                     
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Aliases ┃ Genome digest                    ┃ Asset group   ┃ Asset   ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ t7      │ kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB │ fasta         │ default │
│ t7      │ kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB │ bowtie2_index │ 2.3.5   │
└─────────┴──────────────────────────────────┴───────────────┴─────────┘

One of the assets was also archived (a neccessary step to serve the assets via the refgenie server). Let's list the archived assets.

In [18]:
print(refgenie.stage.table())

                                                   Staged Assets                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━┓
┃ Asset Digest                                    ┃ Asset name                                      ┃ Mode ┃ Size ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━┩
│ 2b4c1900ec5a2b9fce39e45a71e6bed38ca234d394b8af… │ kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index… │ file │ -    │
└─────────────────────────────────────────────────┴─────────────────────────────────────────────────┴──────┴──────┘

Asset `bowtie2_index` has been built successfully for the `t7` genome, and automatically tagged with `2.3.5`, indicating the version of Bowtie2 software used (this behavior is encoded in the recipe).

## Interact with aliases

Let's list the aliases:

In [19]:
refgenie.alias.table()

                Genome aliases                
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Aliases ┃ Genome digest                    ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rCRSd   │ jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP │
│ t7      │ kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB │
└─────────┴──────────────────────────────────┘

Let's assign another alias to the same genome digest, this way we can refer to the same genome in multiple ways.

In [20]:
t7_alias = refgenie.set_genome_alias(
    alias_name="Bacteriophage-T7",
    genome_digest="kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB",
    genome_description="My favorite genome",
)
fav_alias = refgenie.set_genome_alias(
    alias_name="myFavGenome",
    genome_digest="kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB",
    genome_description="My favorite genome",
)

INFO     Added alias: Bacteriophage-T7                                                                 ]8;id=855736;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py\alias.py]8;;\:]8;id=936734;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py#305\305]8;;\

INFO     Created alias directories:                                                                 ]8;id=69036;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=915659;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\
         /tmp/refgenie_demo_2hzk45es/alias/Bacteriophage-T7/fasta/default                                          

INFO     Created alias directories:                                                                 ]8;id=656620;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=333929;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\
         /tmp/refgenie_demo_2hzk45es/alias/Bacteriophage-T7/bowtie2_index/2.3.5                                    

INFO     Added alias: myFavGenome                                                                      ]8;id=816539;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py\alias.py]8;;\:]8;id=134430;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py#305\305]8;;\

INFO     Created alias directories: /tmp/refgenie_demo_2hzk45es/alias/myFavGenome/fasta/default     ]8;id=623489;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=539688;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\

INFO     Created alias directories:                                                                 ]8;id=272265;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=254252;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#223\223]8;;\
         /tmp/refgenie_demo_2hzk45es/alias/myFavGenome/bowtie2_index/2.3.5                                         

The new alias should be listed in the aliases:

In [21]:
refgenie.alias.table()

                             Genome aliases                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Aliases                           ┃ Genome digest                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rCRSd                             │ jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP │
│ t7, Bacteriophage-T7, myFavGenome │ kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB │
└───────────────────────────────────┴──────────────────────────────────┘

The command not only creates a new alias, but also creates a symbolic links to the files in the `data` directory for that genome.

Conversely, alias removal will remove the symbolic links, but not the files in the `data` directory.

In [22]:
refgenie.alias.remove("myFavGenome")

INFO     Deleting alias-owned files: /tmp/refgenie_demo_2hzk45es/alias/myFavGenome                  ]8;id=14326;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=494265;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#145\145]8;;\

INFO     Deleting alias-owned files: /tmp/refgenie_demo_2hzk45es/builds/myFavGenome                 ]8;id=876552;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py\symlinks.py]8;;\:]8;id=764736;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/symlinks.py#145\145]8;;\

INFO     Removed alias: myFavGenome                                                                    ]8;id=5969;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py\alias.py]8;;\:]8;id=765699;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/alias.py#337\337]8;;\

## Retrieve paths to assets

Most importantly, we can retrieve paths to refgenie-managed files.

All below commands will return the same path to the fasta file managed by Refgenie:

In [23]:
print(refgenie.asset.seek("t7", "fasta"))
print(refgenie.asset.seek("Bacteriophage-T7", "fasta", "default"))
print(refgenie.asset.seek("t7", "fasta", "default", "fasta"))

/tmp/refgenie_demo_2hzk45es/alias/t7/fasta/default/t7.fa

/tmp/refgenie_demo_2hzk45es/alias/Bacteriophage-T7/fasta/default/Bacteriophage-T7.fa

/tmp/refgenie_demo_2hzk45es/alias/t7/fasta/default/t7.fa

## Remove an asset

Let's remove the `bowtie2_index` asset for the `dm6` genome.

In [24]:
refgenie.asset.remove_group("bowtie2_index", genome_name="t7")


INFO     Queued asset files for removal after commit:                                                 ]8;id=835720;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/events.py\events.py]8;;\:]8;id=89247;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/events.py#108\108]8;;\
         /tmp/refgenie_demo_2hzk45es/data/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index/2b4c1900ec5a              
         2b9fce39e45a71e6bed38ca234d394b8af28c2a3b0e09548d51e                                                      

INFO     Removed directory:                                                                          ]8;id=304700;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/cleanup.py\cleanup.py]8;;\:]8;id=775289;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/cleanup.py#107\107]8;;\
         /tmp/refgenie_archive_demo_3ps6xfvv/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index/2.3.5                  

INFO     Removed empty directory:                                                                     ]8;id=299537;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/events.py\events.py]8;;\:]8;id=756484;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/events.py#185\185]8;;\
         /tmp/refgenie_archive_demo_3ps6xfvv/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index                        

INFO     Removed directory:                                                                          ]8;id=506912;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/cleanup.py\cleanup.py]8;;\:]8;id=664619;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/db/cleanup.py#107\107]8;;\
         /tmp/refgenie_demo_2hzk45es/data/kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index/2b4c1900ec5               
         a2b9fce39e45a71e6bed38ca234d394b8af28c2a3b0e09548d51e                                                     

INFO     Removed asset group and all assets 'kN9XHLKLS_u7ei2GH87H-qpQrkz8moPB/bowtie2_index'         ]8;id=110123;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/manager.py\manager.py]8;;\:]8;id=990074;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset/manager.py#568\568]8;;\

## Data channels

Refgenie supports data channels, which are used to allow third-party tool developers to expose their recipes and asset classes to Refgenie ecosystem.
In the simplest case, data channels is just a github repository with an index file that lists available asset classes and recipes, like so:

```yaml
asset_class:
  dir: asset_classes # optional, needed only if the asset classes are stored in a subdirectory
  files: # list of asset class files, relative to the index file (or directory)
    - fasta.yaml 
    - bowtie2_index.yaml
recipe:
  dir: recipes # optional, needed only if the recipes are stored in a subdirectory
  files: # list of recipe files, relative to the index file (or directory)
    - fasta.yaml
    - bowtie2_index.yaml
```

One such example is the [refgenie/recipes](https://github.com/refgenie/recipes/blob/refgenie1/index.yaml) repository, which can be added as a data channel to refgenie in the following way:

In [25]:
from refgenie.db.tables import DataChannelType

data_channel = refgenie.sources.add_channel(
    name="refgenie-recipes",
    type=DataChannelType.https,
    index_address="https://refgenie.github.io/refgenie-registry/index.yaml",
    description="Refgenie recipes channel",
)

print(refgenie.sources.channels_table())

                                                   Data Channels                                                   
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Name             ┃ Type  ┃ Index Address                           ┃ Description              ┃ Credentials set ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ refgenie-recipes │ https │ https://refgenie.github.io/refgenie-re… │ Refgenie recipes channel │ False           │
└──────────────────┴───────┴─────────────────────────────────────────┴──────────────────────────┴─────────────────┘

Subsequently, the asset classes and recipes from the data channel can be listed and added to the refgenie instance.

In [26]:
for asset_class in refgenie.sources.iter_asset_classes("refgenie-recipes"):
    try:
        refgenie.asset_class.add(asset_class)
    except Exception as e:
        print(e)

for recipe in refgenie.sources.iter_recipes("refgenie-recipes"):
    try:
        refgenie.recipe.add(recipe)
    except Exception as e:
        print(e)
        


INFO     HTTP Request: GET https://refgenie.github.io/refgenie-registry/index.yaml "HTTP/1.1 200    ]8;id=110854;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=988638;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         OK"                                                                                                       

INFO     Reading YAML from URL:                                                                            ]8;id=265341;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=290895;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\
         https://refgenie.github.io/refgenie-registry/asset_classes/abundant_sequences.yaml                        

INFO     HTTP Request: GET                                                                          ]8;id=969654;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=103640;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://refgenie.github.io/refgenie-registry/asset_classes/abundant_sequences.yaml                        
         "HTTP/1.1 200 OK"                                                                                         

INFO     Registered 'abundant_sequences' asset class                                              ]8;id=415894;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py\asset_class.py]8;;\:]8;id=935835;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/asset_class.py#90\90]8;;\

INFO     Reading YAML from URL: https://refgenie.github.io/refgenie-registry/asset_classes/bed.yaml        ]8;id=279724;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=647618;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\

...

INFO     HTTP Request: GET                                                                          ]8;id=71763;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=963442;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         https://refgenie.github.io/refgenie-registry/recipes/tallymer_index/recipe.yaml "HTTP/1.1                 
         200 OK"                                                                                                   

INFO     Registered 'tallymer_index' recipe                                                           ]8;id=158337;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py\recipe.py]8;;\:]8;id=474862;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py#140\140]8;;\

INFO     Reading YAML from URL: https://refgenie.github.io/refgenie-registry/recipes/tgMap/recipe.yaml     ]8;id=682946;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py\io.py]8;;\:]8;id=369448;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/utils/io.py#31\31]8;;\

INFO     HTTP Request: GET https://refgenie.github.io/refgenie-registry/recipes/tgMap/recipe.yaml   ]8;id=529003;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=91436;file:///home/nsheff/.local/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
         "HTTP/1.1 200 OK"                                                                                         

INFO     Registered 'tgMap' recipe                                                                    ]8;id=169354;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py\recipe.py]8;;\:]8;id=204093;file:///home/nsheff/Dropbox/workspaces/refgenie/repos/refgenie1/refgenie/managers/recipe.py#140\140]8;;\

Alternatively, the same can be achieved by running the following CLI command:

```bash
refgenie1 data_channel sync refgenie-recipes --exists-ok
```

## RefgetStore sequence collection comparison

Refgenie uses RefgetStore (from gtars) for sequence collection operations. Under the hood, refgenie uses the SeqCol digests to uniquely identify genomes. You can use RefgetStore directly to compute digests and compare sequence collections.

In [27]:
from gtars.refget import RefgetStore

store = RefgetStore.in_memory()
store.set_quiet(True)
d1, _ = store.add_sequence_collection_from_fasta(
    REFGENIE_CODE_PATH.parent / "tests/data/rCRSd.fa"
)
d2, _ = store.add_sequence_collection_from_fasta(
    REFGENIE_CODE_PATH.parent / "tests/data/rCRSd-extra.fa"
)
store.compare(d1.digest, d2.digest)

{'digests': {'a': 'jthDpfNIgzM5AGJlOkRtfnky4rXMBIUP',
  'b': 'smiTbD3jP5EwF4DNWVm0c6DGKRlFHfas'},
 'attributes': {'a_only': [],
  'b_only': [],
  'a_and_b': ['lengths',
   'name_length_pairs',
   'names',
   'sequences',
   'sorted_name_length_pairs',
   'sorted_sequences']},
 'array_elements': {'a_count': {'sorted_name_length_pairs': 1,
   'sequences': 1,
   'sorted_sequences': 1,
   'lengths': 1,
   'names': 1,
   'name_length_pairs': 1},
  'b_count': {'sequences': 2,
   'names': 2,
   'sorted_name_length_pairs': 2,
   'sorted_sequences': 2,
   'name_length_pairs': 2,
   'lengths': 2},
  'a_and_b_count': {'sorted_name_length_pairs': 1,
   'sorted_sequences': 1,
   'name_length_pairs': 1,
   'names': 1,
   'sequences': 1,
   'lengths': 1},
  'a_and_b_same_order': {'lengths': None,
   'name_length_pairs': None,
   'sorted_sequences': None,
   'names': None,
   'sequences': None,
   'sorted_name_length_pairs': None}}}